# Libisevate keskmiste optimeerimine
Laadi üles selle projekti ZIP. Käivita lahtrid järjekorras. Sama engine.py töötab veebilehel. Näites ei hangita värskeid hindu: kasutatakse ZIP-i andmeid ja kuvatakse nende kuupäev.

In [ ]:
!pip -q install numpy pandas scipy matplotlib
from google.colab import files
from zipfile import ZipFile
from pathlib import Path
import sys, json
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
with ZipFile(archive) as z:
    z.extractall("ma_project")
root = Path("ma_project")
sys.path.insert(0, str(root / "dist"))
from engine import run
sys.path.insert(0, str(root / "scripts"))
from walk_forward import walk_forward

## 1. Andmed ja kontrollitavad reeglid
Muuda sümboliks MSFT või NVDA. Puutetsoon on 1%, sisenemine toimub pärast kinnitatud signaali järgmise päeva avamisel. Otsime treeningus, testime eraldi.

In [ ]:
symbol = "AAPL"
data = json.loads((root / "dist" / "data" / f"{symbol}.json").read_text())
print(data["source"])
params = dict(start=data["prices"][-756]["date"], min_period=5, max_period=100, step=5, tolerance=1, train=0.7, horizon=20)
result = run({**data, "params": params, "action": "optimize"})
import pandas as pd
pd.DataFrame([dict(type=r["spec"]["type"], period=r["spec"]["period"], score=r["score"], train=r["train"]["20"]["mean"], test=r["test"]["20"]["mean"], test_n=r["test"]["20"]["n"]) for r in result["ranking"]])

## 2. Parima treeningtulemuse pilt
Varjutus on sõltumatu test. Visuaalselt meeldiv tulemus ei ole prognoos.

In [ ]:
import matplotlib.pyplot as plt
best = result["ranking"][0]
dates = pd.to_datetime(result["dates"])
plt.figure(figsize=(14,5))
plt.plot(dates, [r["close"] for r in result["prices"]], label=symbol, alpha=.7)
plt.plot(dates, best["ma"], label=str(best["spec"]))
plt.fill_between(dates, best["lower"], best["upper"], alpha=.15)
plt.axvspan(pd.Timestamp(result["split_date"]), dates[-1], color="grey", alpha=.12)
plt.legend(); plt.grid(alpha=.2); plt.show()

## 3. Walk-forward
Iga testiosa ette valitakse parameetrid uuesti varasemate hindade põhjal. See katse võib võtta veidi aega.

In [ ]:
folds = walk_forward(data["prices"][-1008:], initial=504, test_size=126, params=dict(min_period=10,max_period=60,step=10))
pd.DataFrame([dict(start=f["test_start"], end=f["test_end"], ma=str(f["spec"]), n=f["test"]["20"]["n"], mean=f["test"]["20"]["mean"]) for f in folds])

## Iseseisev katse
1. Muuda tolerants 1% → 1,4%. Kas puuteid tuleb rohkem?
2. Muuda kulu 5 → 20 baaspunkti. Mis juhtub netotulemusega?
3. Miks ei või valida TOP 5 hulgast parimat testitootlust ja nimetada seda sõltumatuks kontrolliks?

**Vastuse suunis:** laiem tsoon võib puuteid ka liita, seega sündmuste arv ei pea kasvama. Suurem kulu vähendab iga sama tehingu tulemust, kuid võib muuta otsingu võitjat. Testi järgi valimine muudab testi treeningu osaks; lõplikuks kontrolliks on vaja uut ajavahemikku.